# 🤖 RAG-Based Customer Support Assistant
## Built with LangGraph, ChromaDB, Groq LLM & HITL
---
**Project:** RAG Internship Project
**Knowledge Base:** Notes on AIML
**Tools:** LangChain · ChromaDB · LangGraph · Groq · Sentence Transformers

In [ ]:
!pip install langchain langchain-community langchain-groq chromadb sentence-transformers langgraph pypdf reportlab -q

print("✅ All libraries installed successfully!")

In [ ]:
!pip install langchain-huggingface -q
print("✅ Done!")
!pip install langchain-text-splitters -q
print("✅ Done!")

✅ Done!
✅ Done!


## 📦 Step 2: Imports & Configuration
Setting up all libraries and API keys needed for the project.

In [ ]:
# ============================================================
# CELL 2: Imports & Configuration (Fixed v3)
# ============================================================

# --- Standard Libraries ---
import os
import warnings
warnings.filterwarnings("ignore")

# --- LangChain Core ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# --- Embeddings ---
from langchain_huggingface import HuggingFaceEmbeddings

# --- Vector Store ---
from langchain_community.vectorstores import Chroma

# --- Groq LLM ---
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# --- LangGraph ---
from langgraph.graph import StateGraph, END
from typing import TypedDict, Optional

# --- API Key ---
os.environ["GROQ_API_KEY"] = "# 🔑 Replace with your key"

# --- Quick Test ---
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
response = llm.invoke("Say hello in one sentence.")
print("✅ Imports done!")
print("🤖 LLM Test:", response.content)

✅ Imports done!
🤖 LLM Test: Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


## 📄 Step 3: Load PDF & Chunk It
Loading the AI_detailed_notes PDF, splitting it into chunks for embedding.

In [ ]:
# ============================================================
# CELL 3: Upload PDF, Load & Chunk It (FIXED VERSION)
# ============================================================

# --- Step 1: Upload PDF ---
from google.colab import files
uploaded = files.upload()

pdf_path = list(uploaded.keys())[0]
print(f"📁 Uploaded file: {pdf_path}")

# --- Step 2: Install required libraries ---
!pip install -q langchain langchain-community langchain-text-splitters pypdf

# --- Step 3: Imports (UPDATED) ---
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Step 4: Load PDF ---
loader = PyPDFLoader(pdf_path)
pages = loader.load()

print("✅ PDF Loaded!")
print(f"📄 Total pages: {len(pages)}")

# --- Step 5: Chunking ---
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(pages)

print(f"✂️ Total chunks created: {len(chunks)}")

# --- Step 6: Preview ---
print("\n🔍 Preview of first 3 chunks:")
print("=" * 60)

for i, chunk in enumerate(chunks[:3]):
    print(f"\n📌 Chunk {i+1}:")
    print(chunk.page_content[:500])
    print("-" * 40)

Saving AI_Detailed_Notes.pdf to AI_Detailed_Notes.pdf
📁 Uploaded file: AI_Detailed_Notes.pdf
✅ PDF Loaded!
📄 Total pages: 1
✂️ Total chunks created: 3

🔍 Preview of first 3 chunks:

📌 Chunk 1:
Artificial Intelligence - Detailed Notes
1. Introduction
Artificial Intelligence (AI) is a branch of computer science that enables machines to simulate human
intelligence such as learning, reasoning, and problem-solving.
2. Types of AI
Narrow AI: Designed for specific tasks.
General AI: Can perform any intellectual task like humans.
Super AI: Hypothetical AI surpassing human intelligence.
3. Machine Learning
----------------------------------------

📌 Chunk 2:
3. Machine Learning
Machine Learning is a subset of AI that allows systems to learn from data without explicit
programming. It includes supervised, unsupervised, and reinforcement learning.
4. Deep Learning
Deep Learning uses neural networks with multiple layers to model complex patterns in data. It is
widely used in image and speech recogn

## 🧠 Step 4: Embed Chunks & Store in ChromaDB
Converting text chunks into vector embeddings and storing them in ChromaDB for retrieval.

In [ ]:
# ============================================================
# CELL 4: Embed Chunks & Store in ChromaDB
# ============================================================

# --- Step 1: Load Embedding Model ---
print("⏳ Loading embedding model (first time may take 1-2 mins)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("✅ Embedding model loaded!")

# --- Step 2: Store in ChromaDB ---
print("\n⏳ Storing chunks in ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="/content/drive/My Drive/chroma_db"
)
print("✅ ChromaDB created and saved to Google Drive!")

# --- Step 3: Test Retrieval ---
print("\n🔍 Testing retrieval with sample query...")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

test_query = "What is AI?"
results = retriever.invoke(test_query)

print(f"\n📌 Query: '{test_query}'")
print(f"📦 Retrieved {len(results)} chunks:")
print("=" * 60)
for i, doc in enumerate(results):
    print(f"\n📄 Result {i+1}:")
    print(doc.page_content)
    print("-" * 40)

## 🔗 Step 5: Build LangGraph Workflow
Building the graph-based workflow with nodes, edges, state, conditional routing and HITL escalation.

In [ ]:
# ============================================================
# CELL 5: LangGraph Workflow with RAG + Routing + HITL
# ============================================================

# --- Step 1: Define State ---
class GraphState(TypedDict):
    question: str                  # User's question
    retrieved_chunks: list         # Chunks from ChromaDB
    answer: str                    # LLM generated answer
    confidence: str                # high / low
    escalated: bool                # Was it escalated to human?
    human_response: Optional[str]  # Human agent's response

# ============================================================
# --- Step 2: Define Nodes ---
# ============================================================

# NODE 1: Retrieve relevant chunks from ChromaDB
def retrieve_node(state: GraphState) -> GraphState:
    print("🔍 [Node 1] Retrieving relevant chunks...")
    question = state["question"]
    docs = retriever.invoke(question)
    state["retrieved_chunks"] = [doc.page_content for doc in docs]
    print(f"   ✅ Retrieved {len(docs)} chunks")
    return state

# NODE 2: Generate answer using LLM
def generate_node(state: GraphState) -> GraphState:
    print("🤖 [Node 2] Generating answer with LLM...")
    question = state["question"]
    context = "\n\n".join(state["retrieved_chunks"])

    prompt = ChatPromptTemplate.from_template("""
You are a helpful customer support assistant.
Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer clearly and concisely.
""")

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": question})
    state["answer"] = response.content
    print(f"   ✅ Answer generated")
    return state

# NODE 3: Check confidence / intent routing
def confidence_node(state: GraphState) -> GraphState:
    print("🧭 [Node 3] Checking confidence & routing...")
    answer = state["answer"].lower()

    # Low confidence triggers
    low_confidence_phrases = [
        "i don't have enough information",
        "i don't know",
        "not mentioned",
        "unclear",
        "cannot find",
        "no information"
    ]

    if any(phrase in answer for phrase in low_confidence_phrases):
        state["confidence"] = "low"
        print("   ⚠️  Low confidence detected → will escalate")
    else:
        state["confidence"] = "high"
        print("   ✅ High confidence → will answer directly")

    state["escalated"] = False
    return state

# NODE 4: HITL Escalation — human takes over
def hitl_node(state: GraphState) -> GraphState:
    print("\n🚨 [Node 4 - HITL] Escalating to human agent...")
    print("=" * 60)
    print(f"❓ User Question: {state['question']}")
    print(f"🤖 Bot could not answer confidently.")
    print("=" * 60)

    # Simulate human agent response
    human_input = input("👨‍💼 Human Agent — Type your response: ")
    state["human_response"] = human_input
    state["escalated"] = True
    print("   ✅ Human response recorded")
    return state

# NODE 5: Final output
def output_node(state: GraphState) -> GraphState:
    print("\n📤 [Node 5] Preparing final output...")
    return state

# ============================================================
# --- Step 3: Routing Logic ---
# ============================================================

def route_after_confidence(state: GraphState) -> str:
    if state["confidence"] == "low":
        return "hitl"       # Go to human
    else:
        return "output"     # Go to final output

# ============================================================
# --- Step 4: Build the Graph ---
# ============================================================

workflow = StateGraph(GraphState)

# Add nodes
workflow.add_node("retrieve",   retrieve_node)
workflow.add_node("generate",   generate_node)
workflow.add_node("confidence", confidence_node)
workflow.add_node("hitl",       hitl_node)
workflow.add_node("output",     output_node)

# Add edges
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve",   "generate")
workflow.add_edge("generate",   "confidence")
workflow.add_conditional_edges(
    "confidence",
    route_after_confidence,
    {
        "hitl":   "hitl",
        "output": "output"
    }
)
workflow.add_edge("hitl",   "output")
workflow.add_edge("output", END)

# Compile
app = workflow.compile()
print("✅ LangGraph workflow compiled successfully!")
print("\n📊 Graph Structure:")
print("   retrieve → generate → confidence")
print("                              ↓")
print("                    [high] output → END")
print("                    [low]  hitl → output → END")

✅ LangGraph workflow compiled successfully!

📊 Graph Structure:
   retrieve → generate → confidence
                              ↓
                    [high] output → END
                    [low]  hitl → output → END


## 🧪 Step 6: Full End-to-End Test
Testing the complete RAG pipeline with sample queries including HITL escalation.

In [ ]:
# ============================================================
# CELL 6: Full End-to-End Test
# ============================================================

def run_bot(question: str):
    print("\n" + "=" * 60)
    print(f"💬 USER QUESTION: {question}")
    print("=" * 60)

    # Initial state
    initial_state: GraphState = {
        "question": question,
        "retrieved_chunks": [],
        "answer": "",
        "confidence": "",
        "escalated": False,
        "human_response": None
    }

    # Run the graph
    final_state = app.invoke(initial_state)

    # Display final answer
    print("\n" + "=" * 60)
    print("📋 FINAL RESPONSE:")
    print("=" * 60)

    if final_state["escalated"]:
        print(f"🚨 Status    : Escalated to Human Agent")
        print(f"👨‍💼 Response  : {final_state['human_response']}")
    else:
        print(f"✅ Status    : Answered by Bot")
        print(f"🤖 Answer    : {final_state['answer']}")

    print("=" * 60)


# ============================================================
# --- TEST 1: High Confidence (Bot answers directly) ---
# ============================================================
run_bot("What is AI?")


💬 USER QUESTION: What is AI?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : Artificial Intelligence (AI) is a branch of computer science that enables machines to simulate human intelligence such as learning, reasoning, and problem-solving.


In [ ]:
# ============================================================
# CELL 7: Test HITL Escalation
# ============================================================

# TEST 2: This will trigger HITL (question not in PDF)
run_bot("What are types of AI?")


💬 USER QUESTION: What are types of AI?
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : There are three types of AI: 
1. Narrow AI: Designed for specific tasks.
2. General AI: Can perform any intellectual task like humans.
3. Super AI: Hypothetical AI surpassing human intelligence.


In [ ]:
# ============================================================
# CELL 8: Final Summary Test — Both Paths
# ============================================================

print("🧪 RUNNING FULL SYSTEM TEST")
print("=" * 60)

# --- Test 1: Bot answers (High Confidence) ---
run_bot("Explain Machine Learning")

# --- Test 2: Bot answers (High Confidence) ---
run_bot("Applications of AI in real world")

# --- Test 3: HITL Escalation (Low Confidence) ---
run_bot("What is future scope of AI?")

🧪 RUNNING FULL SYSTEM TEST

💬 USER QUESTION: Explain Machine Learning
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : Machine Learning is a subset of AI that allows systems to learn from data without explicit programming. It includes supervised, unsupervised, and reinforcement learning.

💬 USER QUESTION: Applications of AI in real world
🔍 [Node 1] Retrieving relevant chunks...
   ✅ Retrieved 3 chunks
🤖 [Node 2] Generating answer with LLM...
   ✅ Answer generated
🧭 [Node 3] Checking confidence & routing...
   ✅ High confidence → will answer directly

📤 [Node 5] Preparing final output...

📋 FINAL RESPONSE:
✅ Status    : Answered by Bot
🤖 Answer    : Applications of AI in the real world include:

1. Healthcare: Disease predi